In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

# Simulación del modelo de competencia bancaria con externalidad sistémica

Compara equilibrio privado ($a_{priv}$) con óptimo social ($a_{soc}$), calcula pérdidas $L$,
spreads $S = lambda * eta(g) * L$ y el impuesto Pigouviano $t^*$ que alinea incentivos.

Para esta simulación se ajustan parámetros de prueba al inicio del script según los supuestos/calibraciones.


In [2]:
# ---------------------------
# PARÁMETROS (ajustables)
# ---------------------------
alpha = 1.0      # beneficio marginal por unidad de riesgo (private revenue)
kappa = 1.5      # coef de costo convex (higher => less risk)
psi = 0.1        # costo privado por default (fraction)
delta = 0.05     # efecto competitivo agregado (small positive)
n = 10           # número de bancos (simétricos)
beta = 0.8       # contribución lineal de riesgo a pérdidas agregadas
gamma = 0.02     # contribución cuadrática (correlación/cola)
lam = 1.0        # escala entre costo fiscal y spread (S = lam * eta * L)

# eta(g): sensibilidad fiscal a pérdidas según severidad macro g
# g: 0 (benigno) ... 2 (muy severo). Cambia la parametrización si prefieres.
def eta_of_g(g, eta0=0.5, eta1=1.0):
    return eta0 + eta1 * g

# Grid de severidad macro g
g_grid = np.linspace(0.0, 2.0, 201)  # 0..2

In [3]:
# ---------------------------
# CÁLCULOS
# ---------------------------
results = []

# equilibrio privado simétrico (analítico)
a_priv = (alpha - psi - delta / n) / kappa
A_priv = n * a_priv

for g in g_grid:
    eta = eta_of_g(g)
    # equilibrio social simétrico (resolver analíticamente)
    num = (alpha - psi - delta / n - eta * beta)
    den = (kappa + 2 * gamma * n * eta)
    if den == 0:
        a_soc = 0.0
    else:
        a_soc = num / den
    # truncar a_soc >= 0
    if a_soc < 0:
        a_soc = 0.0
    A_soc = n * a_soc
    # pérdidas agregadas
    L_priv = beta * A_priv + gamma * (A_priv ** 2)
    L_soc  = beta * A_soc  + gamma * (A_soc ** 2)
    # spreads
    S_priv = lam * eta * L_priv
    S_soc  = lam * eta * L_soc
    # impuesto Pigouviano t* que iguala a_priv^t = a_soc: t* = alpha - psi - delta/n - kappa * a_soc
    t_star = alpha - psi - delta / n - kappa * a_soc
    results.append({
        'g': g,
        'eta': eta,
        'a_priv': a_priv,
        'a_soc': a_soc,
        'A_priv': A_priv,
        'A_soc': A_soc,
        'L_priv': L_priv,
        'L_soc': L_soc,
        'S_priv': S_priv,
        'S_soc': S_soc,
        'S_gap': S_priv - S_soc,
        't_star': t_star
    })

df = pd.DataFrame(results)

In [4]:
# ---------------------------
# OUTPUTS: tablas y gráficos
# ---------------------------
outdir = "simulation_outputs"
os.makedirs(outdir, exist_ok=True)

# Tabla resumen para algunos puntos g
points = [0.0, 0.5, 1.0, 1.5, 2.0]
summary = df[df['g'].isin(points)][['g','eta','a_priv','a_soc','S_priv','S_soc','t_star']].round(6)
summary.to_csv(os.path.join(outdir, 'summary_points.csv'), index=False)
print("Resumen (puntos representativos):")
print(summary.to_string(index=False))

Resumen (puntos representativos):
  g  eta   a_priv    a_soc    S_priv   S_soc   t_star
0.0  0.5 0.596667 0.291176  2.742678 1.24949 0.458235
0.5  1.0 0.596667 0.050000  5.485356 0.40500 0.820000
1.0  1.5 0.596667 0.000000  8.228033 0.00000 0.895000
1.5  2.0 0.596667 0.000000 10.970711 0.00000 0.895000
2.0  2.5 0.596667 0.000000 13.713389 0.00000 0.895000


In [5]:
# Guardar dataset completo
df.to_csv(os.path.join(outdir, 'simulation_model_io_full.csv'), index=False)
print(f"Saved CSV to {os.path.join(outdir, 'simulation_model_io_full.csv')}")

Saved CSV to simulation_outputs\simulation_model_io_full.csv


In [6]:
# PLOT 1: a_priv vs a_soc
plt.figure(figsize=(8,4))
plt.plot(df['g'], df['a_priv'], label='a_priv (privado)', linewidth=1.5)
plt.plot(df['g'], df['a_soc'], label='a_soc (social)', linewidth=1.5)
plt.xlabel('Severidad macro g (más alto = peor GaR)')
plt.ylabel('Nivel de riesgo por banco (a)')
plt.title('Equilibrio privado vs óptimo social: a_priv y a_soc')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(outdir, 'a_priv_vs_a_soc.png'), dpi=200)
plt.close()

In [7]:
# PLOT 2: Spread privado vs social
plt.figure(figsize=(8,4))
plt.plot(df['g'], df['S_priv'], label='S_priv (spread privado)', linewidth=1.5)
plt.plot(df['g'], df['S_soc'], label='S_soc (spread social)', linewidth=1.5)
plt.xlabel('Severidad macro g')
plt.ylabel('Spread (unidades arbitrarias)')
plt.title('Spread soberano: privado vs social en función de la severidad macro')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(outdir, 'spread_priv_vs_soc.png'), dpi=200)
plt.close()

In [8]:
# PLOT 3: Brecha S_priv - S_soc
plt.figure(figsize=(8,4))
plt.plot(df['g'], df['S_gap'], label='Brecha S_priv - S_soc', linewidth=1.5)
plt.xlabel('Severidad macro g')
plt.ylabel('Brecha en Spread (privado - social)')
plt.title('Brecha en spreads debido a exceso de riesgo privado')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(outdir, 'spread_gap.png'), dpi=200)
plt.close()

In [9]:
# PLOT 4: impuesto t*
plt.figure(figsize=(8,4))
plt.plot(df['g'], df['t_star'], label='t* (impuesto Pigouviano)', linewidth=1.5)
plt.xlabel('Severidad macro g')
plt.ylabel('Impuesto correctivo t* (unidades)')
plt.title('Impuesto Pigouviano necesario para alinear incentivo privado con óptimo social')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(outdir, 't_star.png'), dpi=200)
plt.close()

print(f"Plots saved under directory: {outdir}")
print("Fin de la simulación. Revisa los archivos generados (CSV y PNG).")


Plots saved under directory: simulation_outputs
Fin de la simulación. Revisa los archivos generados (CSV y PNG).
